# 04 Initial Handmade Diagnostics

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`run_rotation_comparison.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/W02_2026-11-09_11-15/scripts/experiment3_handmade/run_rotation_comparison.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)
- [`standardize_all_cm.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/W02_2026-11-09_11-15/scripts/standardize_all_cm.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'03_experiments/W02_2026-11-09_11-15/scripts/experiment3_handmade/run_rotation_comparison.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'd9a8a709428aec06f8281ab73640d6802e5f7f0f'}, '03_experiments/W02_2026-11-09_11-15/scripts/standardize_all_cm.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '0415572fe0cb53690478fcad36262c59dc606400'}}

## run_rotation_comparison

원본: [03_experiments/W02_2026-11-09_11-15/scripts/experiment3_handmade/run_rotation_comparison.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/W02_2026-11-09_11-15/scripts/experiment3_handmade/run_rotation_comparison.py)  
Git blob: `d9a8a709428aec06f8281ab73640d6802e5f7f0f`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)

EXP-유-003 appendix 8: compare Rotation before/after on current four writers.

Only writer 41 and 44 are rotation-corrected; writer 42 (replacement
Grayscale) and 43 are unchanged. Existing trained EXP-유-003 model joblibs are
loaded without refitting; output contains five-model totals, per-writer
comparisons, 16x16 confusion matrices and row-level paired predictions.

From repository root:
    python -m pip install numpy scikit-learn joblib matplotlib
    OPENBLAS_NUM_THREADS=2 OMP_NUM_THREADS=2 

CLI 인자 (정적 추출): `--model-zip`, `--handmade-dir`, `--out`

In [ ]:
%%archive_source 03_experiments/W02_2026-11-09_11-15/scripts/experiment3_handmade/run_rotation_comparison.py
#!/usr/bin/env python3
"""EXP-유-003 appendix 8: compare Rotation before/after on current four writers.

Only writer 41 and 44 are rotation-corrected; writer 42 (replacement
Grayscale) and 43 are unchanged. Existing trained EXP-유-003 model joblibs are
loaded without refitting; output contains five-model totals, per-writer
comparisons, 16x16 confusion matrices and row-level paired predictions.

From repository root:
    python -m pip install numpy scikit-learn joblib matplotlib
    OPENBLAS_NUM_THREADS=2 OMP_NUM_THREADS=2 MKL_NUM_THREADS=2 \
      python 03_experiments/W02_2026-11-09_11-15/scripts/experiment3_handmade/run_rotation_comparison.py \
      --model-zip '/path/to/W02_Experiment2_5models_complete(1).zip' \
      --handmade-dir 02_data/Hand_made \
      --out 03_experiments/W02_2026-11-09_11-15/results/EXP-유-003_rotation_v4

Requires a trustworthy model archive (joblib deserialization executes code).
Never uses the official Test split and never retrains a model. The script
performs a fresh evaluation; the published 2026-11-09 review originally
reused verified old predictions for unchanged writers, as documented in
rotation_review_provenance.json. Wall-clock runtimes may be long for SVC.
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import json
import shutil
import sys
import tempfile
from collections import Counter
from pathlib import Path
from zipfile import ZipFile

import joblib
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

WRITERS = (41, 42, 43, 44)
LABELS = "0123456789ABCDEF"
MODEL_NAMES = (('svc', 'SVC (RBF)'), ('hgb', 'HistGradientBoosting'),
               ('mlp', 'MLP'), ('extra_trees', 'Extra Trees'), ('knn', 'KNN'))
EXPECTED_SHA256 = {
    41: '0a829172a72787c770139c73de34cc8b751181829ff6fe08ec260247b9be5a2f',
    42: 'f364fb453d05b4fe2a705c2b8a80683b404608c7c257d664e3ec7ac86932d4c5',
    43: '10db20997c727c1bc1f1299f8e89626fadd3ec9618468d6759c86b0d2ddc4a66',
    44: '0e266a36e7b29edfcca0514836b5c383d13939948b4e335d70f61bb40bdc084d',
}


def sha256(path: Path) -> str:
    with path.open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()


def load_data(folder: Path) -> dict[int, tuple[np.ndarray, np.ndarray]]:
    out = {}
    for writer in WRITERS:
        p = folder / f'ml26_{writer}.npz'
        if sha256(p) != EXPECTED_SHA256[writer]:
            raise ValueError(f'Unexpected dataset SHA256: {p}; compare with EXP-유-005')
        with np.load(p, allow_pickle=False) as d:
            if d.files != ['img', 'label', 'label_id']:
                raise ValueError(f'Wrong NPZ keys in {p}: {d.files}')
            img, labels, ids = (d[k] for k in ('img', 'label', 'label_id'))
        if img.shape != (640, 28, 28) or img.dtype != np.uint8:
            raise ValueError(f'Bad image shape/dtype {p}')
        if ids.shape != (640,) or ids.dtype != np.uint8:
            raise ValueError(f'Bad labels {p}')
        if not np.array_equal(ids, np.array([LABELS.index(c) for c in labels], dtype=np.uint8)):
            raise ValueError(f'Label ID mismatch {p}')
        if Counter(labels) != Counter({c: 40 for c in LABELS}):
            raise ValueError(f'Class count mismatch {p}')
        if writer == 42 and np.unique(img).size != 256:
            raise ValueError('Writer 42 must use new 256-level grayscale original')
        out[writer] = (img, ids.astype(np.int64))
    return out


def data_for_stage(writer: int, img: np.ndarray, stage: str) -> np.ndarray:
    if stage == 'before' and writer in (41, 44):
        # Standard NPZ is already CCW-rotated. Inverse it to reconstruct the
        # original pre-correction orientation without interpolation.
        img = np.rot90(img, k=-1, axes=(1, 2))
    return img.reshape(640, 784).astype(np.float32)


def csv_write(path: Path, rows: list[dict]):
    if not rows:
        raise ValueError(f'No rows for {path}')
    with path.open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def confusion_png(path: Path, cm: np.ndarray, name: str, stage: str, kind: str = 'count'):
    # Use the same Viridis renderer as the archived CSV-only regeneration.
    # Never modify model predictions or the confusion-matrix cell values.
    script_dir = str(Path(__file__).resolve().parents[1])
    if script_dir not in sys.path:
        sys.path.insert(0, script_dir)
    from standardize_all_cm import draw

    title = f'{name} | Rotation {stage} | Hand-made (n=2,560)'
    if kind == 'normalized':
        title += ' | Row-normalized (%)'
    draw(cm, title, kind, 'handmade', path)


def load_saved_models(archive: Path, target: Path) -> dict[str, Path]:
    # Only extract required trained models and their scaler class definition.
    with ZipFile(archive) as z:
        for entry in z.infolist():
            if not entry.filename.endswith(('.joblib', 'pixel_transform.py')):
                continue
            dest = (target / entry.filename).resolve()
            if not dest.is_relative_to(target):
                raise ValueError(f'Unsafe path in ZIP: {entry.filename}')
            dest.parent.mkdir(parents=True, exist_ok=True)
            with z.open(entry) as source, dest.open('wb') as output:
                shutil.copyfileobj(source, output)
    scaler = list(target.rglob('pixel_transform.py'))
    if len(scaler) != 1:
        raise ValueError('Model archive must include exactly one pixel_transform.py')
    sys.path.insert(0, str(scaler[0].parent))
    import pixel_transform  # noqa: F401, required for custom saved Pipeline loading
    result = {}
    for key, _ in MODEL_NAMES:
        files = list(target.rglob(f'{key}.joblib'))
        if len(files) != 1:
            raise ValueError(f'Missing or duplicate model: {key}')
        result[key] = files[0]
    return result


def run(args):
    data = load_data(args.handmade_dir)
    args.out.mkdir(parents=True, exist_ok=True)
    summary = []
    by_writer = []
    all_preds = {'before': [], 'after': []}
    per_class = []
    confusion_rows = []
    with tempfile.TemporaryDirectory(prefix='exp003_rotation_') as d:
        files = load_saved_models(args.model_zip, Path(d).resolve())
        for key, name in MODEL_NAMES:
            model = joblib.load(files[key])
            stage_y = {}
            stage_pred = {}
            for stage in ('before', 'after'):
                yp, yt = [], []
                for writer in WRITERS:
                    img, labels = data[writer]
                    # Same image/labels for both stages for writer 42 and 43.
                    if stage == 'after' and writer in (42, 43):
                        # The same model, same X; avoid duplicate expensive predict.
                        this_pred = stage_pred['before'][writer]
                    else:
                        x = data_for_stage(writer, img, stage)
                        this_pred = np.asarray(model.predict(x), dtype=np.int64)
                    if this_pred.shape != (640,):
                        raise ValueError(f'Bad prediction shape {name}, {writer}, {stage}')
                    yp.append(this_pred)
                    yt.append(labels)
                    by_writer.append({
                        'model': name, 'stage': stage, 'writer': writer,
                        'n': 640, 'correct': int((this_pred == labels).sum()),
                        'accuracy': float(accuracy_score(labels, this_pred)),
                        'macro_f1': float(f1_score(labels, this_pred, labels=list(range(16)),
                                                   average='macro', zero_division=0)),
                    })
                    for idx, (true, pred) in enumerate(zip(labels, this_pred)):
                        all_preds[stage].append({'model': name, 'key': key, 'writer': writer,
                            'image_idx_in_writer': idx, 'actual': LABELS[int(true)],
                            'predicted': LABELS[int(pred)], 'correct': int(true == pred)})
                stage_y[stage] = np.concatenate(yt)
                stage_pred[stage] = {w: yp[i] for i, w in enumerate(WRITERS)}
                cm = confusion_matrix(stage_y[stage], np.concatenate(yp), labels=list(range(16)))
                if cm.sum() != 2560 or not np.all(cm.sum(axis=1) == 160):
                    raise AssertionError(f'Confusion Matrix support mismatch {key}, {stage}')
                csv_rows = [{'actual\\pred': LABELS[i], **{LABELS[j]: int(cm[i,j]) for j in range(16)}}
                            for i in range(16)]
                csv_write(args.out / f'confusion_matrix_{key}_{stage}.csv', csv_rows)
                confusion_png(args.out / f'confusion_matrix_{key}_{stage}.png', cm, name, stage)
                confusion_png(args.out / f'confusion_matrix_{key}_{stage}_normalized.png', cm, name, stage, 'normalized')
                for i in range(16):
                    for j in range(16):
                        confusion_rows.append({'model': name, 'stage': stage,
                                               'actual': LABELS[i], 'predicted': LABELS[j],
                                               'count': int(cm[i,j])})
                pred = np.concatenate(yp)
                accuracy = float(accuracy_score(stage_y[stage], pred))
                mf1 = float(f1_score(stage_y[stage], pred, labels=list(range(16)),
                                       average='macro', zero_division=0))
                summary.append({'model': name, 'stage': stage, 'n': 2560,
                                'correct': int((stage_y[stage] == pred).sum()),
                                'accuracy': accuracy, 'macro_f1': mf1})
                for i, c in enumerate(LABELS):
                    tr = stage_y[stage] == i
                    pr = pred == i
                    tp = int((tr & pr).sum())
                    precision = tp / int(pr.sum()) if pr.sum() else 0.
                    recall = tp / int(tr.sum())
                    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.
                    per_class.append({'model': name, 'stage': stage, 'class': c,
                                      'support': int(tr.sum()), 'correct': tp,
                                      'precision': precision, 'recall': recall, 'f1': f1})
            print(f'Finished {name}', flush=True)
    by_model = {}
    for r in summary:
        by_model.setdefault(r['model'], {})[r['stage']] = r
    comparison = []
    for model, stages in by_model.items():
        b, a = stages['before'], stages['after']
        comparison.append({'model': model, 'n': 2560, 'before_correct': b['correct'],
            'after_correct': a['correct'], 'before_accuracy': b['accuracy'],
            'after_accuracy': a['accuracy'],
            'change_percentage_points': (a['accuracy'] - b['accuracy']) * 100,
            'before_macro_f1': b['macro_f1'], 'after_macro_f1': a['macro_f1']})
    writer_comp = []
    for model, _ in [(n,n) for _,n in MODEL_NAMES]:
        for w in WRITERS:
            b = next(x for x in by_writer if x['model'] == model and x['writer'] == w and x['stage'] == 'before')
            a = next(x for x in by_writer if x['model'] == model and x['writer'] == w and x['stage'] == 'after')
            writer_comp.append({'model': model, 'writer': w,
                'before_correct': b['correct'], 'after_correct': a['correct'],
                'before_accuracy': b['accuracy'], 'after_accuracy': a['accuracy'],
                'delta_percentage_points': (a['accuracy'] - b['accuracy'])*100,
                'before_macro_f1': b['macro_f1'], 'after_macro_f1': a['macro_f1']})
    csv_write(args.out / 'rotation_before_after_5models.csv', comparison)
    csv_write(args.out / 'rotation_before_after_by_writer.csv', writer_comp)
    csv_write(args.out / 'rotation_before_predictions.csv', all_preds['before'])
    csv_write(args.out / 'rotation_after_predictions.csv', all_preds['after'])
    csv_write(args.out / 'rotation_confusion_long.csv', confusion_rows)
    csv_write(args.out / 'rotation_per_class.csv', per_class)
    (args.out / 'rotation_reproduction_metadata.json').write_text(json.dumps({
        'data_sha256': EXPECTED_SHA256, 'model_archive_sha256': sha256(args.model_zip),
        'rotation_corrected_writers': [41,44], 'unchanged_writers': [42,43],
        'used_existing_models': True, 'models_retrained': False,
        'comparison': comparison}, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Saved', args.out)


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--model-zip', type=Path, required=True)
    p.add_argument('--handmade-dir', type=Path, default=Path('02_data/Hand_made'))
    p.add_argument('--out', type=Path, default=Path('EXP003_rotation_results'))
    run(p.parse_args())

if __name__ == '__main__':
    main()


## standardize_all_cm

원본: [03_experiments/W02_2026-11-09_11-15/scripts/standardize_all_cm.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments/W02_2026-11-09_11-15/scripts/standardize_all_cm.py)  
Git blob: `0415572fe0cb53690478fcad36262c59dc606400`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/03_experiments)

Regenerate W02 baseline and rotation confusion-matrix PNGs from saved CSVs only.

No training, inference or edits to source CSVs. Existing Count PNGs are refreshed
and matching Row-normalized PNGs are added with the approved Viridis design.

CLI 인자 (정적 추출): `--w02`, `--check-only`

In [ ]:
%%archive_source 03_experiments/W02_2026-11-09_11-15/scripts/standardize_all_cm.py
#!/usr/bin/env python3
"""Regenerate W02 baseline and rotation confusion-matrix PNGs from saved CSVs only.

No training, inference or edits to source CSVs. Existing Count PNGs are refreshed
and matching Row-normalized PNGs are added with the approved Viridis design.
"""
from __future__ import annotations

import argparse
import csv
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import numpy as np

LABELS = tuple('0123456789ABCDEF')
BASE_MODELS = {
    'SVC (RBF)': ('SVC_RBF', 'SVC (RBF)'),
    'HistGradientBoostingClassifier': ('HistGradientBoosting', 'HistGradientBoosting'),
    'MLPClassifier': ('MLPClassifier', 'MLPClassifier'),
    'ExtraTreesClassifier': ('ExtraTrees', 'ExtraTreesClassifier'),
    'KNeighborsClassifier': ('KNeighbors', 'KNeighborsClassifier'),
}
ROT_MODELS = {
    'svc': 'SVC (RBF)', 'hgb': 'HistGradientBoosting',
    'mlp': 'MLP', 'extra_trees': 'Extra Trees', 'knn': 'KNN',
}
plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'figure.facecolor': 'white',
    'axes.facecolor': 'white', 'savefig.facecolor': 'white',
    'font.size': 9, 'axes.labelsize': 11, 'axes.titlesize': 13,
    'axes.titleweight': 'semibold', 'xtick.labelsize': 9, 'ytick.labelsize': 9,
})


def label_index(text: str) -> int:
    return LABELS.index(str(text).strip())


def baseline_matrices(path: Path):
    matrices = {name: np.full((16, 16), -1, dtype=np.int64) for name in BASE_MODELS}
    with path.open(encoding='utf-8-sig', newline='') as stream:
        for row in csv.DictReader(stream):
            name = row['model']
            if name not in matrices:
                raise ValueError(f'Unexpected baseline model: {name}')
            i, j = label_index(row['actual_label']), label_index(row['predicted_label'])
            if matrices[name][i, j] != -1:
                raise ValueError(f'Duplicate baseline cell: {name}, {i}, {j}')
            matrices[name][i, j] = int(row['count'])
    for name, cm in matrices.items():
        if np.any(cm < 0) or cm.shape != (16, 16) or int(cm.sum()) != 19200:
            raise ValueError(f'Invalid baseline matrix: {name}')
        if int(cm.max()) > 1400 or np.any(cm.sum(axis=1) <= 0):
            raise ValueError(f'Unexpected baseline class distribution: {name}')
    return matrices


def rotation_matrix(path: Path) -> np.ndarray:
    with path.open(encoding='utf-8-sig', newline='') as stream:
        reader = csv.reader(stream)
        header = next(reader)
        if header != ['actual\\pred', *LABELS]:
            raise ValueError(f'Unexpected rotation header: {path}: {header}')
        rows = list(reader)
    if len(rows) != 16 or [row[0] for row in rows] != list(LABELS):
        raise ValueError(f'Unexpected rotation rows: {path}')
    cm = np.array([[int(x) for x in row[1:]] for row in rows], dtype=np.int64)
    if cm.shape != (16, 16) or np.any(cm < 0) or not np.all(cm.sum(axis=1) == 160):
        raise ValueError(f'Unexpected rotation count distribution: {path}')
    return cm


def luminance(rgba) -> float:
    return float(np.dot(rgba[:3], [0.2126, 0.7152, 0.0722]))


def draw(cm: np.ndarray, title: str, kind: str, dataset: str, destination: Path):
    if kind == 'count' and dataset == 'official':
        # Keep original log(1+count) colour transform for subtle off-diagonal errors.
        shown = np.log1p(cm.astype(float))
        norm = Normalize(vmin=0, vmax=np.log1p(1400))
        ticks = np.array([0, 1, 5, 20, 100, 500, 1000, 1400], dtype=float)
        tick_positions = np.log1p(ticks)
        bar_label = 'Number of images (log colour scale)'
    elif kind == 'count':
        shown = cm.astype(float)
        norm = Normalize(vmin=0, vmax=160)
        tick_positions = [0, 40, 80, 120, 160]
        ticks = None
        bar_label = 'Number of images'
    else:
        shown = cm / cm.sum(axis=1, keepdims=True) * 100.0
        norm = Normalize(vmin=0, vmax=100)
        tick_positions = [0, 25, 50, 75, 100]
        ticks = None
        bar_label = 'Percentage of true class (%)'

    fig, ax = plt.subplots(figsize=(10, 9), constrained_layout=True)
    cmap = matplotlib.colormaps['viridis']
    im = ax.imshow(shown, cmap=cmap, norm=norm, interpolation='nearest', aspect='equal')
    ax.set_xticks(np.arange(16), labels=LABELS)
    ax.set_yticks(np.arange(16), labels=LABELS)
    ax.tick_params(axis='both', which='major', length=0, pad=6)
    ax.set_xlabel('Predicted label', labelpad=10)
    ax.set_ylabel('True label', labelpad=9)
    ax.set_title(title, pad=14)
    ax.set_xticks(np.arange(-.5, 16, 1), minor=True)
    ax.set_yticks(np.arange(-.5, 16, 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=.38, alpha=.31)
    ax.tick_params(which='minor', bottom=False, left=False)
    for side in ('left', 'right', 'top', 'bottom'):
        ax.spines[side].set_visible(False)

    for i in range(16):
        for j in range(16):
            if cm[i, j] == 0:
                continue
            v = cm[i, j] if kind == 'count' else shown[i, j]
            value_text = str(int(v)) if kind == 'count' else f'{v:.1f}'
            fg = '#14202e' if luminance(cmap(norm(shown[i, j]))) > .53 else '#ffffff'
            ax.text(j, i, value_text, ha='center', va='center', fontsize=6.15,
                    weight='medium', color=fg)

    cb = fig.colorbar(im, ax=ax, fraction=.045, pad=.026, shrink=.89, ticks=tick_positions)
    if ticks is not None:
        cb.ax.set_yticklabels([str(int(v)) for v in ticks])
    cb.set_label(bar_label, rotation=90, labelpad=10, fontsize=9)
    cb.outline.set_visible(False)
    destination.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(destination, dpi=300, facecolor='white', bbox_inches='tight', pad_inches=.14)
    plt.close(fig)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--w02', type=Path, default=Path(__file__).resolve().parents[1],
                        help='W02 experiment directory containing results/')
    parser.add_argument('--check-only', action='store_true', help='Validate CSVs without writing images')
    args = parser.parse_args()
    results = args.w02 / 'results'
    baseline = baseline_matrices(results / 'EXP-유-003_confusion_long.csv')
    products = []
    for model, (stem, shown_name) in BASE_MODELS.items():
        cm = baseline[model]
        folder = results / 'confusion_matrices'
        for kind, suffix in (('count', ''), ('normalized', '_normalized')):
            products.append((cm, f'{shown_name} | Official Validation (n=19,200)' +
                             (' | Row-normalized (%)' if kind == 'normalized' else ''),
                             kind, 'official', folder / f'{stem}_confusion_matrix{suffix}.png'))
    folder = results / 'EXP-유-003_rotation_v4'
    for model, shown_name in ROT_MODELS.items():
        for stage in ('before', 'after'):
            cm = rotation_matrix(folder / f'confusion_matrix_{model}_{stage}.csv')
            for kind, suffix in (('count', ''), ('normalized', '_normalized')):
                products.append((cm, f'{shown_name} | Rotation {stage} | Hand-made (n=2,560)' +
                                 (' | Row-normalized (%)' if kind == 'normalized' else ''),
                                 kind, 'handmade',
                                 folder / f'confusion_matrix_{model}_{stage}{suffix}.png'))
    if len(products) != 30:
        raise AssertionError(f'Expected 30 outputs, got {len(products)}')
    print('Validated original CSVs; 15 Count + 15 row-normalized PNG outputs', flush=True)
    if args.check_only:
        return
    for index, (matrix, title, kind, dataset, dest) in enumerate(products, 1):
        draw(matrix, title, kind, dataset, dest)
        print(f'{index:02d}/30 {dest}', flush=True)


if __name__ == '__main__':
    main()


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")